# Preprocesamiento

Apartado del enunciado: **3.1**, mínimo tres técnicas de preprocesamiento, indicando qué problemática
resuelve cada una.

## Objetivos

1. Saber **cómo afecta cada preprocesado a los modelos**, con un ajuste mínimo de hiperparámetros (el
   ajuste completo se hace en el modelado).
2. Saber **si imputar los nulos ayuda o no a Random Forest**, que puede trabajar con NaN directamente.

## Estructura del cuaderno

El cuaderno está pensado para que **añadir un preprocesado nuevo sea solo añadir una línea**:

1. **Configuración**: librerías, semilla y umbrales (todos salen de `01_eda.ipynb`).
2. **Carga del dataset** y resumen de nulos.
3. **Catálogo de pasos**: cada paso es una función `df -> df` que imprime lo que ha hecho.
   Están agrupados en: eliminar filas, valores imposibles a NaN, incoherencias cruzadas, columnas y pasos
   opcionales.
4. **Registro de preprocesados** (`PREPROCESADOS`): cada preprocesado es una lista de pasos.
5. **Codificación, imputación, escalado y modelos**: se hacen dentro de un `Pipeline`, que se ajusta solo
   con el train de cada partición para evitar fugas de información.
6. **Evaluación**: todos los preprocesados se prueban con los mismos modelos y las mismas particiones.
7. **Resultados**, incluido el experimento de imputación.

| ID | Pasos que añade | Problemática del EDA que resuelve |
| :--- | :--- | :--- |
| `P0_base` | Ninguno (solo codificación y mediana del pipeline) | Ninguna. Es la base de comparación |
| `P1_limpieza` | Limpieza común | Menores, valores imposibles, incoherencias entre variables, ejercicio con decimales, `colesterol_total` corrupto |
| `P2_diabetes` | P1 + recodificar `diabetes_mellitus` | 597 diabéticos por criterio analítico sin marcar |
| `P3_pcr` | P2 + eliminar PCR > 10 mg/L | PCR de proceso agudo, que no refleja riesgo crónico |

## 1. Configuración

Se fija la **semilla** y se reúnen los **umbrales de dominio** en una sola celda. Si se cambia un umbral,
se cambia aquí y afecta a todos los preprocesados.

In [78]:
from pathlib import Path
import sklearn
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

In [79]:
pd.set_option("display.max_columns", None, "display.width", 200)

SEED = 123
TARGET = "riesgo_cardiovascular"

# Ruta al dataset: funciona desde notebooks/, desde notebooks/workspace/ y desde la raíz del proyecto
RUTAS = ["../data/raw/enfermedad_cardiovascular.csv",
         "../../data/raw/enfermedad_cardiovascular.csv",
         "data/raw/enfermedad_cardiovascular.csv"]
DATA_PATH = next(Path(r) for r in RUTAS if Path(r).exists())

# --- Umbrales del EDA (01_eda.ipynb) ---
EDAD_MINIMA = 18           # el enunciado limita la población a adultos
IMC_MINIMO = 13            # por debajo de 12-13 es biológicamente imposible (mínimo en los datos: 3.6)
EJERCICIO_MAXIMO = 7       # días por semana
HBA1C_MINIMA = 3           # HbA1c < 3 % no es fisiológica
SISTOLICA_MINIMA = 60      # mmHg, ver 01_eda: "Mínimos plausibles de la presión arterial"
DIASTOLICA_MINIMA = 40     # mmHg, ver 01_eda: "Mínimos plausibles de la presión arterial"
PCR_MAXIMA = 10            # mg/L; por encima indica un proceso agudo, no riesgo crónico
HBA1C_DIABETES = 6.5       # % (criterio diagnóstico de diabetes)
GLUCOSA_DIABETES = 126     # mg/dL en ayunas (criterio diagnóstico de diabetes)

# --- Variables categóricas ---
NOMINALES = ["tipo_alimentacion", "antecedentes_familiares_tipo"]
ORDINAL = ["nivel_actividad"]
ORDEN_ACTIVIDAD = ["Sedentario", "Ligeramente activo", "Moderadamente activo", "Muy activo"]
# --- Variables booleanas ---
BOOLEANOS = ['fumador_actual', 'hipertension_previa', 'diabetes_mellitus', 'tratamiento_estatinas', 'uso_antihipertensivos']
# --- Variables numéricas ---
NUMERICAS = ['edad', 'presion_sistolica', 'presion_diastolica', 'colesterol_total',  'colesterol_hdl', 'colesterol_ldl', 'trigliceridos', 'glucosa_ayunas', 'indice_masa_corporal', 'frecuencia_cardiaca_reposo', 'horas_sueno_diarias', 'consumo_alcohol_semanal', 'anos_fumador', 'proteina_c_reactiva', 'acido_urico', 'clearance_creatinina', 'frecuencia_ejercicio_semanal', 'perimetro_cintura', 'hemoglobina_glicosilada']

## 2. Carga del dataset

El CSV se lee sin modificarlo. `resumen_nulos` se reutiliza después para ver cuántos NaN deja cada
preprocesado.

In [80]:
def resumen_nulos(df):
    print("Valores nulos:", df.isnull().sum().sum())
    resumen = pd.DataFrame({
        "Nulos": df.isnull().sum(),
        "Porcentaje (%)": (df.isnull().mean() * 100).round(2),
    })
    return resumen[resumen["Nulos"] > 0].sort_values(by="Nulos", ascending=False)


df = pd.read_csv(DATA_PATH)
print(df.shape)
resumen_nulos(df)

(2254, 28)
Valores nulos: 584


,Nulos,Porcentaje (%)
horas_sueno_diarias,180,7.99
proteina_c_reactiva,157,6.97
colesterol_ldl,135,5.99
glucosa_ayunas,112,4.97


## 3. Catálogo de pasos

Cada paso recibe un `DataFrame` y devuelve otro, sin modificar el original. Todos usan dos funciones
auxiliares para que el mensaje que imprimen tenga siempre el mismo formato.

Son **reglas fijas** que no aprenden nada de los datos (no calculan medias ni medianas), así que se pueden
aplicar antes de partir en train y test sin provocar fugas de información.

In [81]:
def eliminar_filas(df, mascara, motivo):
    antes = len(df)
    df = df[~mascara]
    print(f"{antes - len(df)} filas eliminadas por {motivo}. Hay {len(df)} filas restantes.")
    return df


def poner_nan(df, mascara, columnas, motivo):
    df = df.copy()
    df.loc[mascara, columnas] = np.nan
    print(f"{int(mascara.sum())} valores con {motivo} se han puesto a NaN en {', '.join(columnas)}.")
    return df

### 3.1 Eliminar filas

- **Menores de edad** (4): el enunciado limita la población a adultos.
- **Más años fumando que edad** (1): el registro entero es incoherente.

In [82]:
def eliminar_menores(df):
    return eliminar_filas(df, df["edad"] < EDAD_MINIMA, "ser menores de edad")


def eliminar_fumador_incoherente(df):
    return eliminar_filas(df, df["anos_fumador"] > df["edad"], "tener más años fumando que edad")

### 3.2 Valores imposibles a NaN

Un valor imposible es un error de registro, pero el resto del paciente puede ser válido. Por eso la celda
pasa a **NaN** y la fila se conserva; después la imputa el pipeline (sección 5).

Las reglas están en una tabla (`REGLAS_RANGO`): para añadir una nueva basta con añadir una línea.
Los mínimos de la presión arterial están justificados con referencias clínicas en `01_eda.ipynb`.

**La presión invertida (diastólica >= sistólica) se comprueba antes que los mínimos.** Si fuera al revés,
un paciente con sistólica 52,2 y diastólica 90,6 perdería primero la sistólica, y la comparación
`90,6 >= NaN` daría `False`: su diastólica, probablemente errónea, se conservaría. Comprobándola antes,
las dos presiones pasan a NaN, igual que en el resto de casos invertidos.

In [83]:
def nan_presion_invertida(df):
    return poner_nan(df, df["presion_diastolica"] >= df["presion_sistolica"],
                     ["presion_sistolica", "presion_diastolica"], "presión diastólica >= sistólica")


# (columna, condición que hace imposible el valor, descripción)
REGLAS_RANGO = [
    ("colesterol_hdl", lambda s: s <= 0, "colesterol_hdl negativo o 0"),
    ("trigliceridos", lambda s: s <= 0, "trigliceridos negativos o 0"),
    ("consumo_alcohol_semanal", lambda s: s < 0, "consumo de alcohol semanal negativo"),
    ("indice_masa_corporal", lambda s: s < IMC_MINIMO, f"IMC menor que {IMC_MINIMO}"),
    ("frecuencia_ejercicio_semanal", lambda s: s > EJERCICIO_MAXIMO, f"más de {EJERCICIO_MAXIMO} días de ejercicio semanal"),
    ("hemoglobina_glicosilada", lambda s: s < HBA1C_MINIMA, f"hemoglobina glicosilada menor que {HBA1C_MINIMA}%"),
    ("presion_sistolica", lambda s: s < SISTOLICA_MINIMA, f"presión sistólica menor que {SISTOLICA_MINIMA} mmHg"),
    ("presion_diastolica", lambda s: s < DIASTOLICA_MINIMA, f"presión diastólica menor que {DIASTOLICA_MINIMA} mmHg"),
]

def nan_valores_imposibles(df):
    for columna, condicion, motivo in REGLAS_RANGO:
        df = poner_nan(df, condicion(df[columna]), [columna], motivo)
    return df

### 3.3 Incoherencias entre variables a NaN

Cuando dos variables se contradicen no se sabe cuál de las dos está mal, así que **ambas** pasan a NaN.
La presión invertida es también una incoherencia de este tipo, pero se trata antes, en la sección 3.2.

| Incoherencia | Por qué es un error |
| :--- | :--- |
| IMC > 35 y cintura < 70 cm | Obesidad grado II con cintura de persona delgada |
| IMC < 18,5 y cintura > 110 cm | Bajo peso con cintura de obesidad abdominal |
| HbA1c < 5,5 % y glucosa > 180 mg/dL | HbA1c normal con glucosa de diabetes descompensada |
| HbA1c > 9 % y glucosa < 80 mg/dL | HbA1c de diabetes mal controlada con glucosa normal-baja |

In [84]:
# (condición sobre el DataFrame, columnas que pasan a NaN, descripción)
REGLAS_CRUZADAS = [
    (lambda d: (d["indice_masa_corporal"] > 35) & (d["perimetro_cintura"] < 70),
     ["indice_masa_corporal", "perimetro_cintura"], "IMC mayor que 35 y cintura menor que 70"),
    (lambda d: (d["indice_masa_corporal"] < 18.5) & (d["perimetro_cintura"] > 110),
     ["indice_masa_corporal", "perimetro_cintura"], "IMC menor que 18.5 y cintura mayor que 110"),
    (lambda d: (d["hemoglobina_glicosilada"] < 5.5) & (d["glucosa_ayunas"] > 180),
     ["hemoglobina_glicosilada", "glucosa_ayunas"], "hemoglobina glicosilada menor que 5.5% y glucosa en ayunas mayor que 180"),
    (lambda d: (d["hemoglobina_glicosilada"] > 9) & (d["glucosa_ayunas"] < 80),
     ["hemoglobina_glicosilada", "glucosa_ayunas"], "hemoglobina glicosilada mayor que 9% y glucosa en ayunas menor que 80"),
]


def nan_incoherencias_cruzadas(df):
    for condicion, columnas, motivo in REGLAS_CRUZADAS:
        df = poner_nan(df, condicion(df), columnas, motivo)
    return df

### 3.4 Columnas

**Las variables que cuentan unidades enteras se redondean.** El EDA detectó decimales en variables que
deberían ser enteras:

| Variable | Con decimales | Redondeo | Motivo |
| :--- | :--- | :--- | :--- |
| `edad` | 2.042 | Hacia abajo (45,8 → 45) | La edad son años cumplidos |
| `frecuencia_ejercicio_semanal` | 81,5 % | Hacia arriba (6,1 → 7) | Si ha hecho ejercicio algún rato de un día, ese día cuenta |
| `anos_fumador` | 1.588 | Al más cercano | Años de tabaquismo |
| `frecuencia_cardiaca_reposo` | 2.014 | Al más cercano | Latidos por minuto |
| `consumo_alcohol_semanal` | 2.055 | Al más cercano | Unidades de alcohol por semana |

- El redondeo se hace después de eliminar a los menores y de poner a NaN los imposibles. Como la edad se
  redondea hacia abajo, nadie que tuviera 18 años o más baja de 18, y tampoco aparece ningún paciente con
  más años fumando que edad.
- El ejercicio se redondea después de poner a NaN los valores mayores que 7 (sección 3.2), así que el
  resultado nunca pasa de 7. Su media sube alrededor de medio día.
- En el resto, el redondeo no sesga la variable: la media y la correlación con el target apenas cambian
  (menos de 0,01). Los fumadores actuales con 0 años pasan de 123 a 132, porque quien lleva menos de
  medio año fumando queda en 0 (el EDA ya consideraba justificable este caso).

**`colesterol_total` se elimina**: el 24,8 % de los pacientes tiene un total menor que HDL + LDL (imposible),
el 57,76 % no cuadra con la fórmula HDL + LDL + TG/5 y su correlación con el target es muy baja (ver EDA).
Recalcularla con la fórmula no aportaría información nueva, porque sería una combinación lineal de
variables que el modelo ya recibe.

In [85]:
# columna: función de redondeo
REDONDEO = {
    "edad": np.floor,
    "frecuencia_ejercicio_semanal": np.ceil,
    "anos_fumador": np.round,
    "frecuencia_cardiaca_reposo": np.round,
    "consumo_alcohol_semanal": np.round,
}


def redondear_enteros(df):
    df = df.copy()
    for columna, redondeo in REDONDEO.items():
        cambiados = int((df[columna] != redondeo(df[columna])).sum() - df[columna].isna().sum())
        df[columna] = redondeo(df[columna])
        print(f"{cambiados} valores de {columna} se han redondeado ({redondeo.__name__}).")
    return df


def eliminar_colesterol_total(df):
    print("Columna colesterol_total eliminada.")
    return df.drop(columns="colesterol_total")

### 3.5 Pasos opcionales

- **Recodificar diabetes** (muy recomendable): según los criterios diagnósticos, un paciente con
  HbA1c >= 6,5 % o glucosa en ayunas >= 126 mg/dL es diabético. El EDA encontró 597 pacientes que los
  cumplen y no están marcados en `diabetes_mellitus`.
- **Eliminar PCR > 10 mg/L**: una PCR tan alta indica un proceso agudo (infección, inflamación), no riesgo
  cardiovascular crónico, así que se supone que la prueba no es válida. Los pacientes **sin** PCR medida
  (NaN) se conservan. Al eliminar filas, el conjunto de test de este preprocesado es algo menor que el de
  los demás (sección 6).

In [86]:
def recodificar_diabetes(df):
    df = df.copy()
    cond = (df["hemoglobina_glicosilada"] >= HBA1C_DIABETES) | (df["glucosa_ayunas"] >= GLUCOSA_DIABETES)
    nuevos = int((cond & (df["diabetes_mellitus"] == 0)).sum())
    df.loc[cond, "diabetes_mellitus"] = 1
    print(f"{int(cond.sum())} pacientes con hemoglobina glicosilada >= {HBA1C_DIABETES}% o glucosa en ayunas "
          f">= {GLUCOSA_DIABETES} mg/dL se han puesto como diabéticos ({nuevos} no lo estaban).")
    return df


def eliminar_pcr_alta(df):
    return eliminar_filas(df, df["proteina_c_reactiva"] > PCR_MAXIMA,
                          f"tener proteína C reactiva mayor que {PCR_MAXIMA} mg/L")

## 4. Registro de preprocesados

Cada preprocesado es una **lista de pasos** que se aplican en orden. Cada uno añade **un solo cambio** al
anterior, así que la diferencia de métricas entre dos preprocesados consecutivos se puede atribuir a ese
cambio.

### Cómo añadir un preprocesado nuevo

1. Si hace falta, escribir el paso en la sección 3 como una función `df -> df`
   (o añadir una línea a `REGLAS_RANGO` / `REGLAS_CRUZADAS`).
2. Añadir una entrada a `PREPROCESADOS`, por ejemplo:

```python
def mi_paso(df):
    ...
    return df

PREPROCESADOS["P4_mi_paso"] = PREPROCESADOS["P3_pcr"] + [mi_paso]
```

3. Si el cambio está en la codificación, la imputación o el escalado, va en `construir_pipeline`
   (sección 5).

El resto del cuaderno (evaluación y resultados) recorre `PREPROCESADOS` y no hay que tocarlo.

In [87]:
LIMPIEZA_COMUN = [
    eliminar_menores,              # Eliminar menores de edad (ver sección 3.1)
    eliminar_fumador_incoherente,  # Eliminar fumadores con años de fumador > edad (ver sección 3.1)
    nan_presion_invertida,         # Eliminar presiones invertidas antes que los mínimos de presión (ver sección 3.2)
    nan_valores_imposibles,        # Eliminar valores científicamente imposibles (ver sección 3.3)
    redondear_enteros,             # Realizar redondeos de columnas con valores decimales a enteros después de poner a NaN los imposibles (ver sección 3.4)
    eliminar_colesterol_total,     # Eliminar la columna de colesterol total (ver sección 3.5)
    nan_incoherencias_cruzadas,    # Eliminar incoherencias entre los valores de diferentes columnas (ver sección 3.6)
]

PREPROCESADOS = {
    "P0_base": [],
    "P1_limpieza": LIMPIEZA_COMUN,
    "P2_diabetes": LIMPIEZA_COMUN + [recodificar_diabetes], # Recodificar diabetes según HbA1c y glucosa en ayunas para indicar casos de diabetes que no estaban claramente definidos (ver sección 3.7)
    "P3_pcr": LIMPIEZA_COMUN + [recodificar_diabetes, eliminar_pcr_alta], # Eliminar pacientes con PCR > 10 mg/L (ver sección 3.8)
}


def aplicar_preprocesado(df, pasos):
    for paso in pasos:
        df = paso(df)
    return df

Se aplica cada preprocesado al dataset completo y se guarda el resultado en `DATOS`. Los mensajes
muestran cuántos valores cambia cada paso.

In [88]:
DATOS = {}
for nombre, pasos in PREPROCESADOS.items():
    print(f"\n----------- {nombre} -----------")
    DATOS[nombre] = aplicar_preprocesado(df, pasos)
    print("Dimensiones:", DATOS[nombre].shape)


----------- P0_base -----------
Dimensiones: (2254, 28)

----------- P1_limpieza -----------
4 filas eliminadas por ser menores de edad. Hay 2250 filas restantes.
0 filas eliminadas por tener más años fumando que edad. Hay 2250 filas restantes.
44 valores con presión diastólica >= sistólica se han puesto a NaN en presion_sistolica, presion_diastolica.
4 valores con colesterol_hdl negativo o 0 se han puesto a NaN en colesterol_hdl.
4 valores con trigliceridos negativos o 0 se han puesto a NaN en trigliceridos.
177 valores con consumo de alcohol semanal negativo se han puesto a NaN en consumo_alcohol_semanal.
19 valores con IMC menor que 13 se han puesto a NaN en indice_masa_corporal.
41 valores con más de 7 días de ejercicio semanal se han puesto a NaN en frecuencia_ejercicio_semanal.
1 valores con hemoglobina glicosilada menor que 3% se han puesto a NaN en hemoglobina_glicosilada.
0 valores con presión sistólica menor que 60 mmHg se han puesto a NaN en presion_sistolica.
3 valores con

**Nulos tras la limpieza común.** Todos estos NaN los imputa el pipeline (sección 5).

In [89]:
resumen_nulos(DATOS["P1_limpieza"])

Valores nulos: 990


,Nulos,Porcentaje (%)
horas_sueno_diarias,180,8.00
consumo_alcohol_semanal,177,7.87
proteina_c_reactiva,157,6.98
colesterol_ldl,134,5.96
glucosa_ayunas,113,5.02
indice_masa_corporal,53,2.36
presion_diastolica,47,2.09
presion_sistolica,44,1.96
frecuencia_ejercicio_semanal,41,1.82
perimetro_cintura,34,1.51


No hay ninguna columna nominal que tenga nulos, tanto en P0 como en P1, por lo que no se tiene que aplicar técnicas como imputar los nulos con la moda antes de realizar transformaciones de las variables.

## 5. Codificación, imputación, escalado y modelos

Se hacen dentro de un `Pipeline`, que se **ajusta solo con el train** de cada partición. Así la mediana y
el escalado no ven el test.

- **Dummies** (`OneHotEncoder`, `drop="first"`) para `tipo_alimentacion` y `antecedentes_familiares_tipo`.
- **Ordinal 0-3** (`OrdinalEncoder`) para `nivel_actividad`, porque sus categorías tienen un orden natural.
- **Mediana** para los nulos, salvo en el experimento de imputación (ver más abajo).
- **StandardScaler** solo para los modelos sensibles a la escala (SVM y KNN).

**Modelos con una regularización mínima.** Con los parámetros por defecto, Random Forest y SVM (con
`C=10`) memorizaban el train (F1 macro de 0,998 a 1) y la diferencia con el test superaba 0,38. En esta
fase solo se comparan preprocesados, así que se reajustan un poco, sin buscar los mejores
hiperparámetros (eso se hace en el modelado):

| Modelo | Antes | Ahora | Efecto |
| :--- | :--- | :--- | :--- |
| Random Forest | Árboles sin límite | `max_depth=8`, `min_samples_leaf=5` | Los árboles no pueden aislar pacientes uno a uno |
| SVM | `C=10` | `C=1` (valor por defecto) | Frontera más suave, penaliza menos los errores de train |

**Experimento de imputación.** Desde scikit-learn 1.4, Random Forest admite NaN: en cada división aprende
hacia qué rama enviar los valores que faltan, así que puede aprovechar que falte un dato como
información. Se añade **`Random Forest (NaN)`**, idéntico a `Random Forest` pero sin imputar. Como solo
cambia la imputación, la diferencia entre los dos mide su efecto. SVM y KNN no admiten NaN, así que
siempre se imputan.

In [90]:
# Común a todos los pipelines de preprocesado
transformadores_categoricos = [
    ("nominales", OneHotEncoder(drop="first", handle_unknown="ignore"), NOMINALES),
    ("ordinal", OrdinalEncoder(categories=[ORDEN_ACTIVIDAD]), ORDINAL),
    ("booleanos", "passthrough", BOOLEANOS)
]

# Preprocesado que imputa valores nulos con la mediana y escala los datos numéricos con escalado estándar (solo numericos)
pipeline_imputacionMediana_escaladoEstandar = ColumnTransformer(
    transformers=[
        ("imputación", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), NUMERICAS)
    ] + transformadores_categoricos
)

# Para dado un pipeline y un modelo, construir un pipeline completo que incluya el preprocesado y el modelo (y opcionalmente un GridSearchCV si se proporciona un param_grid).
def construir_pipeline_evaluacion(pipeline_preprocesado, modelo, param_grid=None):
    pipeline_completo = Pipeline([
        ("preprocesado", pipeline_preprocesado),
        ("modelo", modelo)
    ])
    if param_grid is not None:
        return GridSearchCV(estimator=pipeline_completo, param_grid=param_grid, cv=3, scoring='f1_macro', n_jobs=-1)
    return pipeline_completo


## 6. Evaluación

- **`StratifiedShuffleSplit`** con 3 iteraciones y 30 % de test. Se estratifica porque las clases están
  desbalanceadas (la clase 2 tiene menos de la mitad de ejemplos que la 0).
- Las particiones se calculan **una sola vez sobre el dataset original** y se reutilizan en todos los
  preprocesados. Si un preprocesado elimina filas, simplemente no aparecen en su train ni en su test.
  Así todos se comparan sobre los mismos pacientes.
- **Métricas** de train y de test, para detectar sobreajuste:
  - *accuracy*;
  - *F1 macro*, que pesa igual las tres clases;
  - *recall de alto riesgo* (clase 2), la más crítica clínicamente: un paciente de alto riesgo no
    detectado es el peor error.

In [91]:
N_SPLITS = 3
TEST_SIZE = 0.3

splitter = StratifiedShuffleSplit(n_splits=N_SPLITS, test_size=TEST_SIZE, random_state=SEED)

def metricas(y_real, y_pred, conjunto):
    return {
        f"{conjunto}_accuracy": accuracy_score(y_real, y_pred),
        f"{conjunto}_f1_macro": f1_score(y_real, y_pred, average="macro"),
        f"{conjunto}_recall_alto": recall_score(y_real, y_pred, labels=[2], average="macro"),
    }


def evaluar(pipeline, X, y, nombre_modelo, nombre_preprocesado, verbose=False):
    metricas_folds = {
        "train_accuracy": [], "train_f1_macro": [], "train_recall_alto": [],
        "test_accuracy": [], "test_f1_macro": [], "test_recall_alto": []
    }
    if verbose:
        print(f"Evaluando modelo {nombre_modelo} con preprocesado {nombre_preprocesado}...")
    for iteracion, (train_idx, test_idx) in enumerate(splitter.split(X, y), start=1):
        X_train_fold, X_test_fold = X.iloc[train_idx], X.iloc[test_idx]
        y_train_fold, y_test_fold = y.iloc[train_idx], y.iloc[test_idx]
        pipeline_fold = clone(pipeline)
        pipeline_fold.fit(X_train_fold, y_train_fold)
        y_pred_train = pipeline_fold.predict(X_train_fold)
        y_pred_test = pipeline_fold.predict(X_test_fold)
        m_train = metricas(y_train_fold, y_pred_train, "train")
        m_test = metricas(y_test_fold, y_pred_test, "test")
        for k, v in {**m_train, **m_test}.items():
            metricas_folds[k].append(v)
        if verbose:
            print(f"--- Iteración {iteracion} ---")
            print(f"  F1 Macro   -> Train: {m_train['train_f1_macro']:.4f} | Test: {m_test['test_f1_macro']:.4f}")
            print(f"  Recall Alto -> Train: {m_train['train_recall_alto']:.4f} | Test: {m_test['test_recall_alto']:.4f}")
            print(f"  Accuracy   -> Train: {m_train['train_accuracy']:.4f} | Test: {m_test['test_accuracy']:.4f}")
    calculadas = {}
    for metrica, valores in metricas_folds.items():
        media = np.mean(valores)
        std = np.std(valores, ddof=1)
        calculadas[metrica] = f"{media:.4f} ± {std:.4f}"
    resumen_dict = {
        "preprocesado": nombre_preprocesado,
        "modelo": nombre_modelo,
        "train_f1_macro": calculadas["train_f1_macro"],
        "test_f1_macro": calculadas["test_f1_macro"],
        "train_recall_alto": calculadas["train_recall_alto"],
        "test_recall_alto": calculadas["test_recall_alto"],
        "train_accuracy": calculadas["train_accuracy"],
        "test_accuracy": calculadas["test_accuracy"],
    }
    df_resumen = pd.DataFrame([resumen_dict])
    if verbose:
        print(f"\nResumen global:")
        print(f"  F1 MACRO   -> Train: {resumen_dict['train_f1_macro']} | Test: {resumen_dict['test_f1_macro']}")
        print(f"  RECALL ALTO-> Train: {resumen_dict['train_recall_alto']} | Test: {resumen_dict['test_recall_alto']}")
        print(f"  ACCURACY   -> Train: {resumen_dict['train_accuracy']} | Test: {resumen_dict['test_accuracy']}")
    return df_resumen


In [92]:
X = DATOS["P3_pcr"].drop(columns=[TARGET])
y = DATOS["P3_pcr"][TARGET].astype(int)

NUMERICAS = NUMERICAS.remove('colesterol_total') if 'colesterol_total' in NUMERICAS else NUMERICAS  # Ya no existe en P3_pcr (TEMPORAL)

param_grid_lr = {
    'modelo__C': [0.01, 0.1, 1.0, 10.0],
    'modelo__solver': ['lbfgs']
}

pipeline_imputacionMediana_escaladoEstandar_lr = construir_pipeline_evaluacion(
    pipeline_imputacionMediana_escaladoEstandar,
    LogisticRegression(max_iter=1000, random_state=SEED),
    param_grid_lr
)
res_lr = evaluar(pipeline_imputacionMediana_escaladoEstandar_lr, X, y, "Logistic Regression", "Preprocesado_3", True)

res_lr


Evaluando modelo Logistic Regression con preprocesado Preprocesado_3...
--- Iteración 1 ---
  F1 Macro   -> Train: 0.6587 | Test: 0.6312
  Recall Alto -> Train: 0.6933 | Test: 0.6667
  Accuracy   -> Train: 0.6971 | Test: 0.6756
--- Iteración 2 ---
  F1 Macro   -> Train: 0.6558 | Test: 0.6474
  Recall Alto -> Train: 0.7061 | Test: 0.6889
  Accuracy   -> Train: 0.6932 | Test: 0.6860
--- Iteración 3 ---
  F1 Macro   -> Train: 0.6540 | Test: 0.6767
  Recall Alto -> Train: 0.6709 | Test: 0.7481
  Accuracy   -> Train: 0.6926 | Test: 0.7054

Resumen global:
  F1 MACRO   -> Train: 0.6562 ± 0.0024 | Test: 0.6518 ± 0.0231
  RECALL ALTO-> Train: 0.6901 ± 0.0178 | Test: 0.7012 ± 0.0421
  ACCURACY   -> Train: 0.6943 ± 0.0024 | Test: 0.6890 ± 0.0151


,preprocesado,modelo,train_f1_macro,test_f1_macro,train_recall_alto,test_recall_alto,train_accuracy,test_accuracy
0,Preprocesado_3,Logistic Regression,0.6562 ± 0.0024,0.6518 ± 0.0231,0.6901 ± 0.0178,0.7012 ± 0.0421,0.6943 ± 0.0024,0.6890 ± 0.0151


In [93]:
resultados = pd.concat([evaluar(nombre, datos, verbose=True) for nombre, datos in DATOS.items()],
                       ignore_index=True)

TypeError: evaluar() missing 3 required positional arguments: 'y', 'nombre_modelo', and 'nombre_preprocesado'

## 7. Resultados

Cada métrica se muestra como **media ± desviación típica** de las 3 iteraciones. La desviación indica
cuánto varía el resultado según la partición: si la diferencia entre dos preprocesados es menor que su
desviación, no se puede asegurar que uno sea mejor que el otro.

`gap_f1` es la diferencia de F1 macro entre train y test en cada iteración: cuanto mayor, más sobreajuste.

In [ ]:
resultados["gap_f1"] = resultados["train_f1_macro"] - resultados["test_f1_macro"]
columnas = [c for c in resultados.columns if c.startswith(("train_", "test_", "gap_"))]

agrupado = resultados.groupby(["preprocesado", "modelo"], sort=False)[columnas]
media, desviacion = agrupado.mean(), agrupado.std()


def media_mas_menos(media, desviacion):
    formato = lambda tabla: tabla.apply(lambda columna: columna.map("{:.4f}".format))
    return formato(media) + " ± " + formato(desviacion)


resumen = media_mas_menos(media, desviacion)
resumen

**Comparación directa por métrica de test** (filas: preprocesado, columnas: modelo).

In [ ]:
for metrica in ["test_f1_macro", "test_recall_alto"]:
    print(metrica)
    display(resumen[metrica].unstack("modelo")[list(MODELOS)])

### Experimento: ¿imputar ayuda a Random Forest?

Para cada preprocesado y cada partición se resta la métrica de `Random Forest (NaN)` menos la de
`Random Forest` (con mediana). Se muestra la media ± desviación de esa diferencia: si es positiva, no
imputar es mejor; si el cero cae dentro de la media ± desviación, no hay una diferencia clara.

`P0_base` es el caso más directo, porque solo tiene los nulos originales del dataset. En `P1`–`P3` hay
además los NaN que pone la limpieza.

In [ ]:
metricas_test = ["test_accuracy", "test_f1_macro", "test_recall_alto"]
por_iteracion = resultados.set_index(["preprocesado", "iteracion", "modelo"])[metricas_test].unstack("modelo")
diferencia = (por_iteracion.xs("Random Forest (NaN)", axis=1, level="modelo")
              - por_iteracion.xs("Random Forest", axis=1, level="modelo"))

agrupado_dif = diferencia.groupby("preprocesado", sort=False)
media_mas_menos(agrupado_dif.mean(), agrupado_dif.std())

### Conclusiones

*Pendiente de redactar tras ejecutar el cuaderno.* Teniendo en cuenta la desviación de cada resultado:

- `P1_limpieza` frente a `P0_base`: efecto de corregir los valores imposibles e incoherentes.
- `P2_diabetes` frente a `P1_limpieza`: efecto de recodificar `diabetes_mellitus`.
- `P3_pcr` frente a `P2_diabetes`: efecto de eliminar los pacientes con PCR > 10 mg/L.
- `Random Forest (NaN)` frente a `Random Forest`: si imputar con la mediana ayuda o no.

In [ ]:
# Guardar el dataset preprocesado en un archivo CSV (SEGURAMENTE QUITAR ESTO)
df_preprocesado = aplicar_preprocesado(df, PREPROCESADOS["P3_pcr"])
df_preprocesado.to_csv("../data/processed/enfermedad_cardiovascular_preprocesado.csv", index=False)